# Import Sources
This example shows how to import voltage and current sources. In this example, we are going to

- Download an example board
- Create a configuration file
  - Add a voltage source between two nets
  - Add a current source between two pins
  - Add a current source between two pin groups
  - Add a current source between two coordinates
  - Add a current source to the nearest pin
  - Add distributed sources
- Import the configuration file

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import toml
from pathlib import Path
import tempfile

from ansys.aedt.core.examples.downloads import download_file
from pyedb import Edb

Define constants.

In [2]:
AEDT_VERSION = "2025.1"
NG_MODE = False

Download the example PCB data.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = download_file(source="edb/ANSYS-HSD_V1.aedb", local_path=temp_folder.name)

## Load example layout

In [4]:
edbapp = Edb(file_edb, edbversion=AEDT_VERSION)

PyAEDT INFO: Star initializing Edb 03:56:40.997958


PyAEDT INFO: Logger is initialized in EDB.


PyAEDT INFO: legacy v0.53.0


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


PyAEDT INFO: Database ANSYS-HSD_V1.aedb Opened in 2025.1


PyAEDT INFO: Cell main Opened


PyAEDT INFO: Builder was initialized.


PyAEDT INFO: EDB initialized.Time lapse 0:00:10.163753


## Create an empty dictionary to host all configurations

In [5]:
cfg = dict()

## Add a voltage source between two nets

Keywords

- **name**. Name of the voltage source.
- **Reference_designator**. Reference designator of the component.
- **type**. Type of the source. Supported types are 'voltage', 'current'
- **positive_terminal**. Supported types are 'net', 'pin', 'pin_group', 'coordinates'
  - **contact_radius**. Optional. Set circular equipotential region.
  - **inline**. Optional. When True, contact points are place in a row.
  - **num_of_contact**. Optional. Number of contact points. Default is 1. Applicable only when inline is True.
- **negative_terminal**. Supported types are 'net', 'pin', 'pin_group', 'coordinates'
- **equipotential**. Set equipotential region on pins when True.

In [6]:
voltage_source = {
    "name": "V_SOURCE_5V",
    "reference_designator": "U4",
    "type": "voltage",
    "magnitude": 1,
    "positive_terminal": {"net": "5V", "contact_radius": "1mm"},
    "negative_terminal": {"net": "GND", "contact_radius": "1mm"},
    "equipotential": True,
}

## Add a current source between two pins

In [7]:
current_source_1 = {
    "name": "I_CURRENT_1A",
    "reference_designator": "J5",
    "type": "current",
    "magnitude": 10,
    "positive_terminal": {"pin": "15"},
    "negative_terminal": {"pin": "14"},
}

## Add a current source between two pin groups

In [8]:
pin_groups = [
    {"name": "IC2_5V", "reference_designator": "IC2", "pins": ["8"]},
    {"name": "IC2_GND", "reference_designator": "IC2", "net": "GND"},
]

In [9]:
current_source_2 = {
    "name": "CURRENT_SOURCE_2",
    "type": "current",
    "positive_terminal": {"pin_group": "IC2_5V"},
    "negative_terminal": {"pin_group": "IC2_GND"},
}

## Add a current source between two coordinates

Keywords

- **layer**. Layer on which the terminal is placed
- **point**. XY coordinate the terminal is placed
- **net**. Name of the net the terminal is placed on

In [10]:
current_source_3 = {
    "name": "CURRENT_SOURCE_3",
    "type": "current",
    "equipotential": True,
    "positive_terminal": {"coordinates": {"layer": "1_Top", "point": ["116mm", "41mm"], "net": "5V"}},
    "negative_terminal": {"coordinates": {"layer": "Inner1(GND1)", "point": ["116mm", "41mm"], "net": "GND"}},
}

## Add a current source reference to the nearest pin

Keywords

- **reference_net**. Name of the reference net
- **search_radius**. Reference pin search radius in meter

In [11]:
current_source_4 = {
    "name": "CURRENT_SOURCE_4",
    "reference_designator": "J5",
    "type": "current",
    "positive_terminal": {"pin": "16"},
    "negative_terminal": {"nearest_pin": {"reference_net": "GND", "search_radius": 5e-3}},
}

## Add distributed current sources

Keywords

- **distributed**. Whether to create distributed sources. When set to True, ports are created per pin

In [12]:
sources_distributed = {
    "name": "DISTRIBUTED",
    "reference_designator": "U2",
    "type": "current",
    "distributed": True,
    "positive_terminal": {"net": "5V"},
    "negative_terminal": {"net": "GND"},
}

## Add setups in configuration

In [13]:
cfg["pin_groups"] = pin_groups
cfg["sources"] = [
    voltage_source,
    current_source_1,
    current_source_2,
    current_source_3,
    current_source_4,
    sources_distributed,
]

## Write configuration into as json file

In [14]:
file_json = Path(temp_folder.name) / "edb_configuration.json"
with open(file_json, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Equivalent toml file looks like below 

In [15]:
toml_string = toml.dumps(cfg)
print(toml_string)

[[pin_groups]]
name = "IC2_5V"
reference_designator = "IC2"
pins = [ "8",]

[[pin_groups]]
name = "IC2_GND"
reference_designator = "IC2"
net = "GND"

[[sources]]
name = "V_SOURCE_5V"
reference_designator = "U4"
type = "voltage"
magnitude = 1
equipotential = true

[sources.positive_terminal]
net = "5V"
contact_radius = "1mm"
[sources.negative_terminal]
net = "GND"
contact_radius = "1mm"
[[sources]]
name = "I_CURRENT_1A"
reference_designator = "J5"
type = "current"
magnitude = 10

[sources.positive_terminal]
pin = "15"
[sources.negative_terminal]
pin = "14"
[[sources]]
name = "CURRENT_SOURCE_2"
type = "current"

[sources.positive_terminal]
pin_group = "IC2_5V"
[sources.negative_terminal]
pin_group = "IC2_GND"
[[sources]]
name = "CURRENT_SOURCE_3"
type = "current"
equipotential = true

[sources.positive_terminal.coordinates]
layer = "1_Top"
point = [ "116mm", "41mm",]
net = "5V"
[sources.negative_terminal.coordinates]
layer = "Inner1(GND1)"
point = [ "116mm", "41mm",]
net = "GND"
[[source

## Import configuration into example layout

In [16]:
edbapp.configuration.load(config_file=file_json)
edbapp.configuration.run()

PyAEDT INFO: Updating boundaries finished. Time lapse 0:00:00.015696


PyAEDT INFO: Updating nets finished. Time lapse 0:00:00


PyAEDT INFO: Updating components finished. Time lapse 0:00:00


PyAEDT INFO: Creating pin groups finished. Time lapse 0:00:00.031264


PyAEDT INFO: Placing sources finished. Time lapse 0:00:00.656212


...

PyAEDT INFO: Applying modeler finished. Time lapse 0:00:00.484419


PyAEDT INFO: Placing ports finished. Time lapse 0:00:00


PyAEDT INFO: Placing probes finished. Time lapse 0:00:00


PyAEDT INFO: Applying operations finished. Time lapse 0:00:00


True

## Review

In [17]:
edbapp.siwave.sources

{'V_SOURCE_5V': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x222efe124d0>,
 'I_CURRENT_1A': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x222f2a81960>,
 'CURRENT_SOURCE_2': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144550>,
 'CURRENT_SOURCE_3': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af1449d0>,
 'CURRENT_SOURCE_4': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144a60>,
 'U2_5V_39': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144b50>,
 'U2_5V_40': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144c10>,
 'U2_5V_41': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144cd0>,
 'U2_5V_42': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af145240>,
 'U2_5V_43': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af144370>,
 'U2_5V_44': <pyedb.dotnet.database.edb_data.ports.ExcitationSources at 0x263af1452a0>,
 

## Save and close Edb
The temporary folder will be deleted once the execution of this script is finished. Replace **edbapp.save()** with
**edbapp.save_as("C:/example.aedb")** to keep the example project.

In [18]:
edbapp.save()
edbapp.close()

PyAEDT INFO: EDB file save time: 0.00ms


PyAEDT INFO: EDB file release time: 0.00ms


True